# Gemma-2-9b-it arm: Elicitation Game organisms on the lens matrix

Runtime > Change runtime type > **A100**. Run the cells in order. Fits and results are saved to Drive, so a disconnect loses nothing: rerun the same cell and it resumes.

Before cell 2: accept the licence at https://huggingface.co/google/gemma-2-9b-it and have a read token ready.

In [ ]:
!pip -q install -U "transformers>=4.56" peft datasets accelerate pyyaml wandb git+https://github.com/anthropics/jacobian-lens

In [ ]:
from huggingface_hub import login; login()
from google.colab import drive; drive.mount('/content/drive')
import os; OUT = '/content/drive/MyDrive/jspace'; os.makedirs(OUT, exist_ok=True)
os.environ['OUT'] = OUT
print('outputs go to', OUT)

In [ ]:
!git clone https://github.com/in-a-quantum-world/anthropic-jspace-scheming
%cd anthropic-jspace-scheming
!git log --oneline -1

## Cell 4: the 200 evaluation items (about 10 minutes)

Samples MMLU test items and keeps the first 200 the original model answers correctly. Afterwards copy `data/external/mmlu_gemma9b_items_v1.json` from Drive into the repo and commit it.

In [ ]:
!python src/run_gemma9b_external.py --out-root $OUT build-items --n 200 --seed 0

## Cell 5: gates (about 20 minutes)

Original, locked and circuit-broken models on the 200 items with and without the trigger, McNemar, and 75-item free-form retention. Paste the three result lines to Claude before fitting.

In [ ]:
!python src/run_gemma9b_external.py --out-root $OUT gates

## Cell 6: lens fits (40 to 70 minutes each)

Run in order. If one stops with CUDA out of memory, rerun it with `--dim-batch 4` and note that on its lens card.

In [ ]:
!python src/run_gemma9b_external.py --out-root $OUT fit --target base --dim-batch 8

In [ ]:
!python src/run_gemma9b_external.py --out-root $OUT fit --target locked --dim-batch 8

In [ ]:
!python src/run_gemma9b_external.py --out-root $OUT fit --target cb --dim-batch 8

## Cell 7: readout (about 40 minutes)

First version after the base and locked fits; second version once the circuit-broken fit exists.

In [ ]:
NP = 'hf:neuronpedia/jacobian-lens:gemma-2-9b-it/jlens/Salesforce-wikitext/gemma-2-9b-it_jacobian_lens.pt'
!python src/run_gemma9b_external.py --out-root $OUT readout --models base locked \
    --lens parent_own=$OUT/lens_fits/gemma-2-9b-it-base/lens.pt \
    --lens refit_locked=$OUT/lens_fits/gemma-2-9b-it-locked/lens.pt \
    --lens neuronpedia=$NP

In [ ]:
NP = 'hf:neuronpedia/jacobian-lens:gemma-2-9b-it/jlens/Salesforce-wikitext/gemma-2-9b-it_jacobian_lens.pt'
!python src/run_gemma9b_external.py --out-root $OUT readout --models base locked cb \
    --lens parent_own=$OUT/lens_fits/gemma-2-9b-it-base/lens.pt \
    --lens refit_locked=$OUT/lens_fits/gemma-2-9b-it-locked/lens.pt \
    --lens refit_cb=$OUT/lens_fits/gemma-2-9b-it-cb/lens.pt \
    --lens neuronpedia=$NP

## Afterwards

Copy `results/local/gemma9b_external_gates.json` and `results/local/gemma9b_external_readout.json` from the Drive folder into the repo's `results/local/` and commit. Lens files stay on Drive (log them as W&B artifacts if you want them tracked).